# Big Data Project - Complete Data Analytics Notebook

This notebook analyzes all available project datasets under `datasets/` and `data/`, checks how columns can be combined,
and builds a reusable pipeline-ready fact table.

It is designed to be runnable on this repository as-is, with a **sample mode** enabled by default for very large files.

## What this notebook does

1. Loads and profiles datasets from both `datasets/` and `data/`.
2. Normalizes date, identifier, and count columns across formats (CSV/TXT, comma/semicolon/tab, mixed encodings).
3. Evaluates which datasets can be combined directly (station-level vs date-level).
4. Builds a canonical fact table for downstream modeling.
5. Adds feature-engineering blocks for demand forecasting and anomaly detection.
6. Demonstrates a future-proof enrichment step for weather and holiday datasets.

## Project Data Explanation

Before analysis, understand folder roles:

`data/`:

- Main working storage for project data.
- Contains large/raw source files (like MTA) and processed outputs (`data/processed/...`).
- Think: pipeline working area.

`datasets/`:

- Curated external CSV datasets used as clean input sources.
- Usually smaller and more structured for analysis.
- Think: curated input package.

So:

- `data/` = raw + processed working data
- `datasets/` = curated dataset inputs

In this notebook, we read both, then normalize them into a canonical fact table.

In [ ]:
from __future__ import annotations

from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

plt.style.use("seaborn-v0_8-whitegrid")

In [ ]:
# Paths and execution controls
ROOT = Path("..").resolve()
DATASETS_DIR = ROOT / "datasets"
DATA_DIR = ROOT / "data"
IDF_DIR = DATA_DIR / "Ile_de_france"
MTA_DIR = DATA_DIR / "soroosh_MTA"

# Sample mode keeps notebook fast and memory-safe for local execution.
USE_SAMPLE_MODE = True
MTA_SAMPLE_ROWS = 300_000
MAX_IDF_FILES = 6
MAX_ROWS_PER_IDF_FILE = 250_000

print("ROOT:", ROOT)
print("USE_SAMPLE_MODE:", USE_SAMPLE_MODE)

In [ ]:
def detect_separator(file_path: Path) -> str:
    # Infer delimiter from first line
    header = file_path.open("rb").readline().decode("latin1", errors="ignore")
    counts = {",": header.count(","), ";": header.count(";"), "	": header.count("	")}
    return max(counts, key=counts.get)


def read_table_smart(file_path: Path, nrows: int | None = None) -> pd.DataFrame:
    # Read CSV/TXT with fallback encodings
    sep = detect_separator(file_path)
    encodings = ["utf-8-sig", "utf-8", "latin1", "cp1252", "utf-16"]
    last_error = None
    for enc in encodings:
        try:
            return pd.read_csv(file_path, sep=sep, encoding=enc, nrows=nrows, low_memory=False)
        except Exception as exc:
            last_error = exc
    raise RuntimeError(f"Failed to read {file_path.name}: {last_error}")


def clean_numeric(series: pd.Series) -> pd.Series:
    # Normalize mixed numeric formats: spaces, comma decimals, text labels
    s = series.astype(str).str.strip()
    s = s.str.replace(" ", "", regex=False)
    s = s.str.replace(" ", "", regex=False)
    s = s.str.replace(" ", "", regex=False)
    s = s.str.replace(",", ".", regex=False)
    s = s.str.replace("Less than 5", "2", regex=False)
    s = s.replace({"": np.nan, "nan": np.nan, "None": np.nan, "?": np.nan})
    return pd.to_numeric(s, errors="coerce")


def parse_any_date(series: pd.Series) -> pd.Series:
    # Parse mixed date formats
    return pd.to_datetime(series, errors="coerce", dayfirst=True)

In [ ]:
# Inventory project data files
file_list = sorted(
    list(DATASETS_DIR.glob("*.csv"))
    + list(IDF_DIR.glob("*.csv"))
    + list(IDF_DIR.glob("data-rf-*/*NB_FER*.txt"))
    + list(IDF_DIR.glob("data-rf-*/*NB_FER*.csv"))
    + list(IDF_DIR.glob("data-rf-*/*PROFIL*.txt"))
    + list(IDF_DIR.glob("data-rf-*/*PROFIL*.csv"))
    + list((IDF_DIR / "data-rf-2020" / "data-rf-2020").glob("*.txt"))
    + list(MTA_DIR.glob("*.csv"))
)

inventory = pd.DataFrame(
    {
        "file": [f.name for f in file_list],
        "relative_path": [str(f.relative_to(ROOT)) for f in file_list],
        "size_mb": [round(f.stat().st_size / (1024 ** 2), 2) for f in file_list],
    }
).sort_values("size_mb", ascending=False)

inventory.head(20)

In [ ]:
# Load core datasets (moderate size)
regularities_fr = pd.read_csv(DATASETS_DIR / "Regularities_by_liaisons_Trains_France.csv")
travel_titles = pd.read_csv(DATASETS_DIR / "Travel_titles_validations_in_Paris_and_suburbs.csv")
idfm_surface = read_table_smart(DATASETS_DIR / "idfm_validations_surface.csv")
tgv_monthly = read_table_smart(IDF_DIR / "regularite-mensuelle-tgv-aqst.csv")

core_datasets = {
    "regularities_fr": regularities_fr,
    "travel_titles": travel_titles,
    "idfm_surface": idfm_surface,
    "tgv_monthly": tgv_monthly,
}

profile_rows = []
for name, df in core_datasets.items():
    profile_rows.append(
        {
            "dataset": name,
            "rows": len(df),
            "columns": len(df.columns),
            "missing_pct": round(df.isna().mean().mean() * 100, 2),
        }
    )

pd.DataFrame(profile_rows).sort_values("rows", ascending=False)

In [ ]:
# Normalize key fields for analytics
travel_titles = travel_titles.copy()
travel_titles["date"] = parse_any_date(travel_titles["DATE"])
travel_titles["validations"] = clean_numeric(travel_titles["NB_VALID"])

idfm_surface = idfm_surface.copy()
idfm_surface["date"] = parse_any_date(idfm_surface["JOUR"])
idfm_surface["validations"] = clean_numeric(idfm_surface["NB_VALD"])

regularities_fr = regularities_fr.copy()
regularities_fr["period"] = pd.to_datetime(regularities_fr["Period"], format="%Y-%m", errors="coerce")
regularities_fr["late_trains_arrival"] = clean_numeric(regularities_fr["Number of trains late on arrival"])

tgv_monthly = tgv_monthly.copy()
tgv_monthly["period"] = pd.to_datetime(tgv_monthly["Date"], format="%Y-%m", errors="coerce")
tgv_monthly["planned_trains"] = clean_numeric(tgv_monthly["Nombre de circulations prévues"])

daily_travel = (
    travel_titles.dropna(subset=["date"])
    .groupby("date", as_index=False)["validations"]
    .sum()
    .assign(source="travel_titles_paris")
)

daily_surface = (
    idfm_surface.dropna(subset=["date"])
    .groupby("date", as_index=False)["validations"]
    .sum()
    .assign(source="idfm_surface")
)

monthly_tgv = (
    tgv_monthly.dropna(subset=["period"])
    .groupby("period", as_index=False)["planned_trains"]
    .sum()
    .rename(columns={"period": "date", "planned_trains": "validations"})
    .assign(source="tgv_planned_trains")
)

core_daily = pd.concat([daily_travel, daily_surface, monthly_tgv], ignore_index=True)
core_daily.head()

In [ ]:
# Trend visualization across core datasets
fig, ax = plt.subplots(figsize=(13, 5))
for src, g in core_daily.groupby("source"):
    g = g.sort_values("date")
    ax.plot(g["date"], g["validations"], label=src, linewidth=1.8)
ax.set_title("Daily/Monthly Demand Signals Across Core Datasets")
ax.set_xlabel("Date")
ax.set_ylabel("Count")
ax.legend()
plt.show()

# Top stations in travel_titles dataset
top_stations = (
    travel_titles.groupby("STATION_NAME", as_index=False)["validations"]
    .sum()
    .sort_values("validations", ascending=False)
    .head(15)
)

fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(top_stations["STATION_NAME"], top_stations["validations"])
ax.set_title("Top 15 Stations by Total Validations (Travel Titles)")
ax.set_ylabel("Validations")
ax.tick_params(axis="x", rotation=75)
plt.tight_layout()
plt.show()

In [ ]:
# Load Ile-de-France NB_FER files (daily validations by stop)
def load_idf_nb_files(base_dir: Path, sample_mode: bool = True) -> pd.DataFrame:
    nb_files = sorted(base_dir.glob("data-rf-*/*NB_FER*.txt")) + sorted(base_dir.glob("data-rf-*/*NB_FER*.csv"))
    nb_files += sorted((base_dir / "data-rf-2020" / "data-rf-2020").glob("*NB_FER*.txt"))

    if sample_mode:
        nb_files = nb_files[-MAX_IDF_FILES:]

    frames = []
    for fp in nb_files:
        nrows = MAX_ROWS_PER_IDF_FILE if sample_mode else None
        df = read_table_smart(fp, nrows=nrows)
        df.columns = [c.strip() for c in df.columns]

        date_col = "JOUR" if "JOUR" in df.columns else None
        id_col = "ID_ZDC" if "ID_ZDC" in df.columns else ("ID_REFA_LDA" if "ID_REFA_LDA" in df.columns else ("lda" if "lda" in df.columns else None))
        val_col = "NB_VALD" if "NB_VALD" in df.columns else None

        if date_col is None or val_col is None:
            continue

        out = pd.DataFrame(
            {
                "date": parse_any_date(df[date_col]),
                "station_id": df[id_col].astype(str) if id_col else np.nan,
                "station_name": df.get("LIBELLE_ARRET", np.nan),
                "ticket_category": df.get("CATEGORIE_TITRE", np.nan),
                "validations": clean_numeric(df[val_col]),
                "source_file": fp.name,
            }
        )
        frames.append(out)

    if not frames:
        return pd.DataFrame(columns=["date", "station_id", "station_name", "ticket_category", "validations", "source_file"])
    return pd.concat(frames, ignore_index=True)


idf_nb = load_idf_nb_files(IDF_DIR, sample_mode=USE_SAMPLE_MODE)
idf_nb_profile = {
    "rows": len(idf_nb),
    "stations": idf_nb["station_id"].nunique(dropna=True),
    "min_date": str(idf_nb["date"].min()),
    "max_date": str(idf_nb["date"].max()),
}
idf_nb_profile

In [ ]:
# Load Ile-de-France PROFIL files (hourly profile percentages)
def load_idf_profil_files(base_dir: Path, sample_mode: bool = True) -> pd.DataFrame:
    profil_files = sorted(base_dir.glob("data-rf-*/*PROFIL*.txt")) + sorted(base_dir.glob("data-rf-*/*PROFIL*.csv"))
    profil_files += sorted((base_dir / "data-rf-2020" / "data-rf-2020").glob("*PROFIL*.txt"))

    if sample_mode:
        profil_files = profil_files[-MAX_IDF_FILES:]

    frames = []
    for fp in profil_files:
        nrows = MAX_ROWS_PER_IDF_FILE if sample_mode else None
        df = read_table_smart(fp, nrows=nrows)
        df.columns = [c.strip() for c in df.columns]

        pct_col = "pourc_validations" if "pourc_validations" in df.columns else ("Pourcentage_validations" if "Pourcentage_validations" in df.columns else None)
        id_col = "ID_ZDC" if "ID_ZDC" in df.columns else ("ID_REFA_LDA" if "ID_REFA_LDA" in df.columns else ("lda" if "lda" in df.columns else None))

        if pct_col is None:
            continue

        out = pd.DataFrame(
            {
                "station_id": df[id_col].astype(str) if id_col else np.nan,
                "cat_jour": df.get("CAT_JOUR", np.nan),
                "hour_bin": df.get("TRNC_HORR_60", np.nan),
                "pct_validations": clean_numeric(df[pct_col]),
                "source_file": fp.name,
            }
        )
        frames.append(out)

    if not frames:
        return pd.DataFrame(columns=["station_id", "cat_jour", "hour_bin", "pct_validations", "source_file"])
    return pd.concat(frames, ignore_index=True)


idf_profil = load_idf_profil_files(IDF_DIR, sample_mode=USE_SAMPLE_MODE)
idf_profil.head()

In [ ]:
# Check which columns allow direct combination across datasets
compatibility = pd.DataFrame(
    [
        {
            "dataset": "travel_titles_paris",
            "date": True,
            "station_id": True,
            "line_id": False,
            "hour_bin": False,
            "ticket_category": True,
            "count_metric": True,
            "geo_area": "Ile-de-France",
        },
        {
            "dataset": "idfm_surface",
            "date": True,
            "station_id": False,
            "line_id": True,
            "hour_bin": False,
            "ticket_category": True,
            "count_metric": True,
            "geo_area": "Ile-de-France",
        },
        {
            "dataset": "idf_nb_fer",
            "date": True,
            "station_id": True,
            "line_id": False,
            "hour_bin": False,
            "ticket_category": True,
            "count_metric": True,
            "geo_area": "Ile-de-France",
        },
        {
            "dataset": "idf_profil_fer",
            "date": False,
            "station_id": True,
            "line_id": False,
            "hour_bin": True,
            "ticket_category": False,
            "count_metric": False,
            "geo_area": "Ile-de-France",
        },
        {
            "dataset": "mta_hourly",
            "date": True,
            "station_id": True,
            "line_id": False,
            "hour_bin": True,
            "ticket_category": True,
            "count_metric": True,
            "geo_area": "NYC",
        },
        {
            "dataset": "tgv_regularities",
            "date": True,
            "station_id": True,
            "line_id": False,
            "hour_bin": False,
            "ticket_category": False,
            "count_metric": True,
            "geo_area": "France",
        },
    ]
)

display(compatibility)

matrix_cols = ["date", "station_id", "line_id", "hour_bin", "ticket_category", "count_metric"]
matrix = compatibility.set_index("dataset")[matrix_cols].astype(int)

fig, ax = plt.subplots(figsize=(9, 4))
im = ax.imshow(matrix.values, aspect="auto")
ax.set_xticks(range(len(matrix_cols)))
ax.set_xticklabels(matrix_cols, rotation=45, ha="right")
ax.set_yticks(range(len(matrix.index)))
ax.set_yticklabels(matrix.index)
ax.set_title("Join-Key Compatibility Matrix (1=available)")
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

## Canonical fact table design

To support current analysis and future integration, we standardize all datasets to:

- `date`
- `region`
- `location_id`
- `location_name`
- `metric_type` (`validations`, `ridership`, `delay`, ...)
- `value`
- `source`

In [ ]:
def aggregate_mta_daily(mta_csv: Path, sample_mode: bool = True) -> pd.DataFrame:
    usecols = [
        "transit_timestamp",
        "station_complex_id",
        "station_complex",
        "borough",
        "payment_method",
        "fare_class_category",
        "ridership",
        "transfers",
    ]

    read_kwargs = dict(usecols=usecols, low_memory=False)
    if sample_mode:
        read_kwargs["nrows"] = MTA_SAMPLE_ROWS

    mta = pd.read_csv(mta_csv, **read_kwargs)
    mta["date"] = pd.to_datetime(mta["transit_timestamp"], errors="coerce")
    mta["date"] = mta["date"].dt.floor("D")
    mta["ridership"] = pd.to_numeric(mta["ridership"], errors="coerce")
    mta["transfers"] = pd.to_numeric(mta["transfers"], errors="coerce")

    daily = (
        mta.dropna(subset=["date"])
        .groupby(["date", "borough", "station_complex_id", "station_complex"], as_index=False)[["ridership", "transfers"]]
        .sum()
    )
    return daily


mta_path = MTA_DIR / "MTA_Subway_Hourly_Ridership__2020-2024.csv"
mta_daily = aggregate_mta_daily(mta_path, sample_mode=USE_SAMPLE_MODE)
mta_daily.head()

In [ ]:
# Build canonical facts from each source
facts = []

tt_fact = pd.DataFrame(
    {
        "date": travel_titles["date"],
        "region": "Ile-de-France",
        "location_id": travel_titles["ID_REFA_LDA"].astype(str),
        "location_name": travel_titles["STATION_NAME"],
        "metric_type": "validations",
        "value": travel_titles["validations"],
        "source": "travel_titles_paris",
    }
)
facts.append(tt_fact)

sf_fact = pd.DataFrame(
    {
        "date": idfm_surface["date"],
        "region": "Ile-de-France",
        "location_id": idfm_surface.get("ID_GROUPOFLINES", pd.Series([np.nan] * len(idfm_surface))).astype(str),
        "location_name": idfm_surface.get("LIBELLE_LIGNE", pd.Series([np.nan] * len(idfm_surface))),
        "metric_type": "validations",
        "value": idfm_surface["validations"],
        "source": "idfm_surface",
    }
)
facts.append(sf_fact)

if not idf_nb.empty:
    nb_fact = pd.DataFrame(
        {
            "date": idf_nb["date"],
            "region": "Ile-de-France",
            "location_id": idf_nb["station_id"].astype(str),
            "location_name": idf_nb["station_name"],
            "metric_type": "validations",
            "value": idf_nb["validations"],
            "source": "idf_nb_fer",
        }
    )
    facts.append(nb_fact)

mta_fact = pd.DataFrame(
    {
        "date": mta_daily["date"],
        "region": mta_daily["borough"].fillna("NYC"),
        "location_id": mta_daily["station_complex_id"].astype(str),
        "location_name": mta_daily["station_complex"],
        "metric_type": "ridership",
        "value": mta_daily["ridership"],
        "source": "mta_hourly_agg_daily",
    }
)
facts.append(mta_fact)

fact_table = pd.concat(facts, ignore_index=True)
fact_table["date"] = pd.to_datetime(fact_table["date"], errors="coerce").dt.floor("D")
fact_table["value"] = pd.to_numeric(fact_table["value"], errors="coerce")
fact_table = fact_table.dropna(subset=["date", "value"])

print("Fact table rows:", len(fact_table))
fact_table.head()

In [ ]:
daily_fact = (
    fact_table.groupby(["date", "region", "source", "metric_type"], as_index=False)["value"]
    .sum()
    .sort_values(["source", "region", "date"])
)

daily_fact.head(10)

In [ ]:
def add_time_features(df: pd.DataFrame, group_cols=("region", "source", "metric_type")) -> pd.DataFrame:
    out = df.copy().sort_values([*group_cols, "date"])
    out["day_of_week"] = out["date"].dt.dayofweek
    out["is_weekend"] = (out["day_of_week"] >= 5).astype(int)
    out["month"] = out["date"].dt.month
    out["week_of_year"] = out["date"].dt.isocalendar().week.astype(int)

    grouped = out.groupby(list(group_cols))["value"]
    out["lag_1"] = grouped.shift(1)
    out["lag_7"] = grouped.shift(7)
    out["rolling_7_mean"] = grouped.transform(lambda s: s.rolling(7, min_periods=3).mean())
    out["rolling_7_std"] = grouped.transform(lambda s: s.rolling(7, min_periods=3).std())
    out["pct_change_1"] = grouped.pct_change()
    out["zscore_30"] = grouped.transform(
        lambda s: (s - s.rolling(30, min_periods=10).mean()) / s.rolling(30, min_periods=10).std()
    )
    return out


featured = add_time_features(daily_fact)
featured.tail()

In [ ]:
selected = featured[featured["source"].isin(["travel_titles_paris", "idfm_surface", "mta_hourly_agg_daily"])]

weekday_profile = (
    selected.groupby(["source", "day_of_week"], as_index=False)["value"].mean()
    .rename(columns={"value": "avg_value"})
)

fig, ax = plt.subplots(figsize=(10, 5))
for src, g in weekday_profile.groupby("source"):
    ax.plot(g["day_of_week"], g["avg_value"], marker="o", label=src)
ax.set_title("Average Demand by Day of Week")
ax.set_xlabel("Day of week (0=Mon)")
ax.set_ylabel("Average count")
ax.legend()
plt.show()

anomaly_view = selected.dropna(subset=["zscore_30"]).copy()
anomaly_view["is_anomaly"] = (anomaly_view["zscore_30"].abs() >= 2.5)
anomaly_rate = (
    anomaly_view.groupby("source", as_index=False)["is_anomaly"].mean()
    .rename(columns={"is_anomaly": "anomaly_rate"})
)
anomaly_rate

## Future integration pipeline: Weather + Holidays

The following block shows how to enrich the canonical fact table with external datasets.
For now we use synthetic examples with the same schema expected from real providers.

In [ ]:
date_span = pd.date_range(daily_fact["date"].min(), daily_fact["date"].max(), freq="D")

holiday_demo = pd.DataFrame(
    {
        "date": date_span,
        "country": np.where(date_span.month <= 6, "FR", "US"),
        "is_holiday": ((date_span.day == 1) & (date_span.month.isin([1, 5, 7, 11]))).astype(int),
        "holiday_name": np.where((date_span.day == 1) & (date_span.month == 1), "New Year", ""),
    }
)

rng = np.random.default_rng(42)
weather_demo = pd.DataFrame(
    {
        "date": np.repeat(date_span, 2),
        "country": ["FR", "US"] * len(date_span),
        "mean_temp_c": rng.normal(loc=14, scale=9, size=len(date_span) * 2),
        "precip_mm": np.clip(rng.gamma(shape=1.8, scale=2.0, size=len(date_span) * 2), 0, 35),
    }
)


def attach_external_context(df: pd.DataFrame, holidays: pd.DataFrame, weather: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["country"] = np.where(out["region"].str.contains("Ile|France", case=False, na=False), "FR", "US")
    out = out.merge(holidays, on=["date", "country"], how="left")
    out = out.merge(weather, on=["date", "country"], how="left")
    out["is_holiday"] = out["is_holiday"].fillna(0).astype(int)
    return out


enriched = attach_external_context(featured, holiday_demo, weather_demo)
enriched[["date", "region", "source", "value", "is_holiday", "mean_temp_c", "precip_mm"]].head()

In [ ]:
# Pipeline blueprint visualization
fig, ax = plt.subplots(figsize=(12, 4))
ax.axis("off")

boxes = [
    (0.05, 0.55, "Raw data\n(datasets/ + data/)"),
    (0.26, 0.55, "Normalization\n(schema + types)"),
    (0.47, 0.55, "Canonical fact table\n(date, region, metric, value)"),
    (0.68, 0.55, "Enrichment\n(weather + holidays)"),
    (0.86, 0.55, "Models\nforecast + anomalies"),
]

for x, y, label in boxes:
    ax.text(
        x,
        y,
        label,
        ha="center",
        va="center",
        fontsize=10,
        bbox=dict(boxstyle="round,pad=0.5", fc="#e8f0fe", ec="#2c3e50"),
    )

for i in range(len(boxes) - 1):
    x1, y1, _ = boxes[i]
    x2, y2, _ = boxes[i + 1]
    ax.annotate("", xy=(x2 - 0.06, y2), xytext=(x1 + 0.08, y1), arrowprops=dict(arrowstyle="->", lw=1.8))

ax.set_title("Big Data Project Pipeline (Current + Future Extensions)")
plt.show()

In [ ]:
# Persist outputs for future modeling scripts
processed_dir = ROOT / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

daily_fact.to_csv(processed_dir / "daily_fact_table.csv", index=False)
enriched.to_csv(processed_dir / "daily_fact_table_enriched_demo.csv", index=False)

print("Saved:")
print("-", processed_dir / "daily_fact_table.csv")
print("-", processed_dir / "daily_fact_table_enriched_demo.csv")

## Key conclusions

- **Best direct joins inside Ile-de-France:** `travel_titles` + `idf_nb_fer` (date + station id).
- **`idfm_surface`** is best merged at **date/line level**, not station level.
- **MTA** is highly valuable for high-frequency modeling, but it should stay in a separate geographic branch (`US`) and join with shared external context (weather/holiday/calendar).
- The canonical fact table created here is ready for forecasting and anomaly pipelines.

To run full scale processing, set:

```python
USE_SAMPLE_MODE = False
```